> **Estado de validación:** Ejecutado completo con Ollama y TinyLlama local; respuestas y errores del modelo analizados.

# Introducción a los LLMs Locales con Ollama

¡Es hora de ponerse prácticos!

Este notebook demuestra cómo ejecutar un Modelo de Lenguaje Grande (LLM) de forma local usando Ollama y LangChain. Todo correrá directamente en tu computador.

Ollama nos permite ejecutar LLMs localmente en nuestra máquina. LangChain es un SDK (biblioteca de código reutilizable) que facilita la interacción con LLMs.

![Alt text](figs/01/ollama_langchain_architecture.svg)
## 1. Prerrequisitos

Antes de ejecutar este notebook, necesitas instalar Ollama y descargar un modelo. Sigue los pasos a continuación.

### 1.1 Instalación de Ollama

**macOS:**
- Descarga el instalador desde [ollama.ai](https://ollama.ai)
- Abre el archivo descargado y sigue las instrucciones de instalación

**Windows:**
- Descarga el instalador desde [ollama.ai](https://ollama.ai)
- Ejecuta el instalador y sigue las instrucciones

**Linux:**

In [1]:
import os, shutil, subprocess, time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
OLLAMA_BIN = os.getenv("OLLAMA_BIN") or shutil.which("ollama")
if not OLLAMA_BIN:
    raise RuntimeError("Instala Ollama o configura OLLAMA_BIN con la ruta a su ejecutable.")
print("Ollama disponible")

Ollama disponible


Verifica tu instalación (Windows, Mac y Linux):

In [2]:
import requests
try:
    requests.get("http://127.0.0.1:11434/api/version", timeout=3).raise_for_status()
    print("Se reutiliza el servidor local de Ollama")
except requests.RequestException:
    ollama_server = subprocess.Popen([OLLAMA_BIN, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            requests.get("http://127.0.0.1:11434/api/version", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            time.sleep(1)
    else:
        raise RuntimeError("No inició el servidor de Ollama")
print(subprocess.check_output([OLLAMA_BIN, "--version"], text=True))

Se reutiliza el servidor local de Ollama
ollama version is 0.40.0



### 1.2 Descarga del Modelo TinyLlama

Descarga el modelo TinyLlama (aproximadamente 600MB). Existe un amplio catálogo de modelos disponibles en Ollama, incluyendo DeepSeek y otros, pero TinyLlama es el más pequeño, lo cual es ideal para propósitos de demostración:


In [3]:
subprocess.run([OLLAMA_BIN, "pull", "tinyllama:1.1b"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("tinyllama:1.1b disponible")

tinyllama:1.1b disponible


## 2. Configuración del Entorno Python

Instala los paquetes necesarios. Instalaremos LangChain, una de las bibliotecas más populares para interactuar con LLMs. langchain-ollama nos permite interactuar con modelos localmente.

In [4]:
# Instaladas desde ../requirements-ejemplos-llm.txt
import langchain_ollama

## 3. Usando el LLM con LangChain

El código a continuación nos permite interactuar con el modelo tinyllama alojado localmente en nuestros computadores. Simplemente enviamos el siguiente mensaje al modelo: "I love programming".

Lee los comentarios del código si quieres entender exactamente qué está pasando.

In [5]:
# Importamos las bibliotecas necesarias
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage

# Inicializamos el LLM. Aquí elegimos el modelo y la temperatura
# (que controla la aleatoriedad de la salida). llm es lo que usaremos para llamar al modelo.
llm = ChatOllama(
    model="tinyllama:1.1b",
    num_predict=512,
    temperature=1,  # 0 para salidas más deterministas
)

# Debemos proveer un arreglo de mensajes al modelo. El primer mensaje es siempre el mensaje
# de sistema, que le indica al modelo cómo comportarse. El segundo es el mensaje del usuario,
# que es lo que queremos preguntarle al modelo.
messages = [
    (
        "system",
        "You are a helpful assistant",
    ),
    ("human", "I love programming."),
]

# Enviamos el arreglo de mensajes al modelo y obtenemos la respuesta.
# La respuesta es un objeto AIMessage que contiene el texto generado, que imprimiremos a continuación.
ai_msg = llm.invoke(messages)

# Mostramos la respuesta
print(ai_msg.content)

I'm glad to hear that you love programming! Here are a few resources that can help you gain practical skills and knowledge in the field.

1. Codecademy: an online coding platform that offers courses in a variety of programming languages and concepts, from basic HTML and CSS to more advanced topics like web development frameworks and databases.

2. CodePen: a platform where coders can share their work, allowing others to browse and explore their projects.

3. Khan Academy: a non-profit educational organization that provides free, self-paced online courses in math, science, history, and other subjects.

4. Udacity: an online education platform that offers a range of courses in data science, machine learning, and other cutting-edge technologies.

5. Datastax: a cloud-based data platform that provides solutions for big data and analytics.

6. DataCamp: an online educational platform for data analysis and visualization.

7. Udacity: an online education platform that offers free courses in c

# 4. Probando Diferentes Prompts

Probemos un par de ejemplos para ver qué puede hacer el modelo, cambiando el arreglo de mensajes:

In [6]:
# Pedimos una explicación
messages = [
    (
        "system",
        "You are a helpful and informative AI assistant.",
    ),
    ("human", "Explain the concept of a neural network in simple terms."),
]

ai_msg = llm.invoke(messages)
print(ai_msg.content)

In simple terms, a neural network is a mathematical model designed to mimic the ways in which the human brain processes information and makes decisions. Here's an example to explain how neural networks work:

Let's say you have a list of items to buy. You want to know which ones are the most popular and the most affordable. The neural network is designed to analyze the data and make predictions based on previous experiences.

Here's an illustration: Suppose we have a dataset of shopping patterns, such as the purchase history of customers. This dataset is used to train a neural network, which creates a model that maps the customer's shopping patterns to the most popular and affordable items. Then, based on this model, the neural network can recommend items to a shopper in the future.

Another example of neural networks is in image recognition, where the network is used to identify objects based on their features, such as the position and color of an object in an image.

In summary, neur

In [7]:
# Pedimos algo de código
messages = [
    (
        "system",
        "You are an expert Python programmer.",
    ),
    ("human", "Write a function to check if a string is a palindrome."),
]

ai_msg = llm.invoke(messages)
print(ai_msg.content)

Here's a function to check if a string is a palindrome in Python:

```python
def is_palindrome(s):
    s = s.lower()
    n = len(s)
    for I in range(n//2):
        if s[i], s[n - i - 1] != s[i]:
            return False
    return True
```

Explanation:

- The `lower()` method converts the string to lowercase and returns the same as the original string.
- The `split()` method splits the string at every newline character and returns a list of strings.
- The length of the list is computed by calculating the number of elements in the list.
- The `range()` method starts at the first element and returns an iterator over the elements. The `+1` is added to count the next element in the list.
- The loop iterates over each element, comparing each string with the one before it and skipping over any non-matches.
- The `if` statement compares each string with the one after it and returns `False` if they're not the same.
- The `else` statement ensures the function always returns `True`.

You can 

## 5. Creando un Chatbot Básico

Vamos a crear una interfaz de chat simple para interactuar con el modelo. Cada celda representará una interacción. Sigue los pasos a continuación; tendrán más sentido a medida que los vayas ejecutando.

In [8]:
# Configuración: ejecuta esta celda primero
from langchain_ollama import ChatOllama

In [9]:
# Inicializamos el LLM
llm = ChatOllama(
    model="tinyllama:1.1b",
    num_predict=512,
    temperature=0
)

# Prompt de sistema: define el comportamiento del asistente
system_prompt = "You are a helpful assistant"

print("-" * 30)

------------------------------


Creemos una celda para nuestra primera interacción, diciéndole al modelo cuál es nuestro color favorito. Ten en cuenta que la respuesta puede ser extraña, ya que no estamos usando el mejor modelo disponible. Pero debería indicar que entendió que tu color favorito es el azul."""

In [10]:
# Primera interacción
user_message = "My favorite color is blue."

# Creamos mensajes nuevos solo para esta interacción (sin historial previo)
messages = [
    ("system", system_prompt),
    ("human", user_message)
]

# Obtenemos la respuesta
ai_msg = llm.invoke(messages)

print(f"Tú: {user_message}")
print(f"Asistente: {ai_msg.content}")

Tú: My favorite color is blue.
Asistente: I do not have the ability to have feelings or preferences like humans do. However, based on your statement, you are a blue-colored individual. Blue is a color that is associated with the sky, water, and the ocean. It is often associated with calmness, tranquility, and serenity. Blue is also a color that is often associated with trust, reliability, and honesty.


Ahora hagamos una pregunta de seguimiento:

In [11]:
# Segunda interacción
user_message = "What's my favorite color?"

# Creamos mensajes nuevos solo para esta interacción (sin historial previo)
messages = [
    ("system", system_prompt),
    ("human", user_message)
]

# Obtenemos la respuesta
ai_msg = llm.invoke(messages)

print(f"Tú: {user_message}")
print(f"Asistente: {ai_msg.content}")

Tú: What's my favorite color?
Asistente: I do not have a favorite color. However, based on your previous responses, you have expressed a preference for blue, green, and yellow. If you're looking for a specific color, you could try asking a friend or family member for their opinion or using a poll on a social media platform.


Como puedes ver, ¡el modelo no recuerda nuestro color favorito! Podrías preguntarte por qué. Seguramente herramientas como ChatGPT pueden recordar el contexto completo de la conversación. Este es el concepto de memoria. El LLM, por sí solo, no recuerda conversaciones pasadas. Necesitamos diseñar nuestra aplicación para que lo haga.

Exploremos cómo hacerlo a continuación, y cómo herramientas como ChatGPT lo gestionan.

## 6. Entendiendo Cómo Funciona la Memoria de Conversación

La forma más sencilla de gestionar esto es pasar todo el historial de conversación al modelo, en lugar de solo el mensaje individual:

In [12]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

# Inicializamos el LLM como antes
llm = ChatOllama(
    model="tinyllama:1.1b",
    num_predict=512,
    temperature=0
)

# Definimos el historial de conversación (siempre empieza con un prompt de sistema).
# conversation es una lista de mensajes. SystemMessage representa un mensaje de sistema
# y acepta el contenido como parámetro.
conversation = [SystemMessage(content="You are a helpful assistant")]

# Definimos una función llamada chat que toma la entrada del usuario como parámetro.
# Esta función se usará para interactuar con el modelo.
def chat(user_input: str):
    # 1) Agregamos el mensaje del usuario al historial.
    # Ahora la lista contiene el mensaje de sistema y el mensaje del usuario.
    conversation.append(HumanMessage(content=user_input))

    # 2) Llamamos al modelo con la lista completa de conversación
    ai_msg = llm.invoke(conversation)

    # 3) Mostramos el intercambio
    print(f"Tú: {user_input}")
    print(f"Asistente: {ai_msg.content}")
    print(f"[Longitud de conversación: {len(conversation) + 1} mensajes]")  # +1 por la respuesta del modelo

    # 4) Agregamos la respuesta del modelo al historial. Ahora la lista contiene el mensaje
    # de sistema, el mensaje del usuario y la respuesta del modelo, que se usará en la próxima interacción.
    conversation.append(ai_msg)

    return ai_msg.content

print("Chat Con Memoria")
print("-" * 30)

Chat Con Memoria
------------------------------


In [13]:
"""Démosle nuevamente nuestro color favorito:"""

# Primera interacción: compartimos el color favorito
chat("My favorite color is blue")

Tú: My favorite color is blue
Asistente: I'm not capable of having favorite colors like humans do. However, based on your description, you mention that you enjoy blue. Here are some examples of how blue can be used in various contexts:

1. Color: blue is a popular color that can be used to create a variety of visual effects, such as:
  - Soft pastel shades: blue can create a soft, pastel-like effect that is perfect for creating a romantic or dreamy atmosphere.
  - Bright and bold: blue can also create a bold and bright effect, which is perfect for creating a sense of excitement or energy.
  - Neutral tones: blue can also be used to create a neutral or muted color palette, which can be used to create a calming or relaxing atmosphere.

2. Color scheme: blue can be used as a color scheme in design, such as:
  - Blue and white: this color scheme is classic and timeless, and can be used to create a clean and elegant look.
  - Blue and green: this color scheme is nature-inspired and can be u

"I'm not capable of having favorite colors like humans do. However, based on your description, you mention that you enjoy blue. Here are some examples of how blue can be used in various contexts:\n\n1. Color: blue is a popular color that can be used to create a variety of visual effects, such as:\n  - Soft pastel shades: blue can create a soft, pastel-like effect that is perfect for creating a romantic or dreamy atmosphere.\n  - Bright and bold: blue can also create a bold and bright effect, which is perfect for creating a sense of excitement or energy.\n  - Neutral tones: blue can also be used to create a neutral or muted color palette, which can be used to create a calming or relaxing atmosphere.\n\n2. Color scheme: blue can be used as a color scheme in design, such as:\n  - Blue and white: this color scheme is classic and timeless, and can be used to create a clean and elegant look.\n  - Blue and green: this color scheme is nature-inspired and can be used to create a calming and rel

In [14]:
"""Ahora veamos si lo recuerda:"""

# Segunda interacción: ponemos a prueba la memoria
chat("What's my favorite color?")

Tú: What's my favorite color?
Asistente: I'm not capable of having favorite colors like humans do. However, based on your description, you mention that you enjoy blue. Here are some examples of how blue can be used in various contexts:

1. Color: blue is a popular color that can be used to create a variety of visual effects, such as:
   - Soft pasteel shades: blue can create a soft, pastel-like effect that is perfect for creating a romantic or dreamy atmosphere.
   - Bright and bold: blue can also create a bold and bright effect, which is perfect for creating a sense of excitement or energy.
   - Neutral or muted tones: blue can also be used to create a neutral or muted color palettte, which can be used to create a calming or relaxing atmosphere.

2. Color scheme: blue can be used as a color scheme in design, such as:
   - Blue and white: this color scheme is classic and timeless, and can be used to create a clean and elegant look.
   - Blue and green: this color scheme is nature-inspi

"I'm not capable of having favorite colors like humans do. However, based on your description, you mention that you enjoy blue. Here are some examples of how blue can be used in various contexts:\n\n1. Color: blue is a popular color that can be used to create a variety of visual effects, such as:\n   - Soft pasteel shades: blue can create a soft, pastel-like effect that is perfect for creating a romantic or dreamy atmosphere.\n   - Bright and bold: blue can also create a bold and bright effect, which is perfect for creating a sense of excitement or energy.\n   - Neutral or muted tones: blue can also be used to create a neutral or muted color palettte, which can be used to create a calming or relaxing atmosphere.\n\n2. Color scheme: blue can be used as a color scheme in design, such as:\n   - Blue and white: this color scheme is classic and timeless, and can be used to create a clean and elegant look.\n   - Blue and green: this color scheme is nature-inspired and can be used to create a

¡Ahora el modelo debería recordar correctamente tu color favorito! Esto se debe a que estamos llevando el registro del historial de conversación.

## 7. El Desafío de la Ventana de Contexto

Aunque nuestra solución de memoria simple funciona inicialmente (porque pasamos todo el historial como entrada), tiene una limitación crítica: **la ventana de contexto**.

### Entendiendo la Ventana de Contexto

Cada modelo de lenguaje tiene una "ventana de contexto" fija: el número máximo de tokens (aproximadamente palabras o fragmentos de palabras) que puede procesar a la vez:

- **TinyLlama**: ~2,048 tokens (aproximadamente 1,500 palabras)
- **GPT-3.5**: ~4,096 tokens
- **GPT-4**: entre 8,192 y 128,000 tokens según la versión
- **Claude 3**: hasta 200,000 tokens

A medida que la conversación crece, eventualmente alcanzamos este límite. Cuando eso ocurre:

1. El modelo no puede ver mensajes más allá del tamaño de la ventana
2. Efectivamente "olvida" las partes más antiguas de la conversación
3. El procesamiento se vuelve más lento con contextos más grandes
4. Puedes recibir errores si superas la ventana de contexto

## 8. Soluciones al Problema de la Ventana de Contexto

En sistemas de producción como ChatGPT, varias técnicas abordan esta limitación:

### 1. Ventana de Contexto Deslizante
Este enfoque conserva solo los N mensajes más recientes, más el prompt de sistema.

**Ejemplo:** Si limitas a 10 mensajes y la conversación llega a 15, descartarías los 5 mensajes más antiguos (excepto el prompt de sistema).

Es como tener una conversación donde solo recuerdas lo dicho en los últimos minutos. Es simple, pero pierde toda la información anterior.

### 2. Resumen de Conversación (Summarization)
Esta técnica condensa las partes más antiguas de la conversación en resúmenes para ahorrar tokens.

**Ejemplo:** Después de 10 intercambios, el sistema podría reemplazar esos 20 mensajes con un único resumen: "El usuario preguntó sobre redes neuronales. El asistente las explicó y proporcionó un ejemplo en Python para detectar palíndromos."

Esto preserva los puntos clave mientras reduce significativamente el uso de tokens.

### 3. Gestión de Sesiones con Base de Datos

Este enfoque usa una base de datos para almacenar historiales completos de conversación asociados a sesiones únicas de usuario. Así funciona:

1. **Creación de sesión**: Cuando un usuario comienza a chatear, el sistema crea un ID de sesión único (como "session_abc123")

2. **Almacenamiento de mensajes**: Cada mensaje del usuario y del modelo se guarda en una tabla de la base de datos vinculada a ese ID de sesión

3. **Gestión de la ventana de contexto**: Al preparar el prompt para el LLM, el sistema:
   - Recupera todos los mensajes de la sesión
   - Aplica estrategias para ajustarse a la ventana de contexto (ventana deslizante o resumen)
   - Envía la conversación optimizada al modelo
   - Almacena la nueva respuesta de vuelta en la base de datos

**Ejemplo práctico:**
- Un usuario ha estado chateando durante horas con ChatGPT
- Su sesión "user_789" ahora tiene 200 mensajes en la base de datos
- Cuando envía el mensaje #201, el sistema:
  - Recupera los 200 mensajes anteriores de la base de datos
  - Selecciona los más relevantes para ajustarse a la ventana de contexto
  - Obtiene una respuesta del modelo
  - Agrega el mensaje #201 y la respuesta a la base de datos
  - Aunque el modelo solo "ve" la porción reciente, el historial completo permanece en la base de datos

Así es como servicios como ChatGPT pueden mantener "memoria" en conversaciones muy largas, incluso cuando cierras el navegador y vuelves más tarde.

## ¿Qué usa ChatGPT típicamente?

ChatGPT usa un enfoque híbrido que combina varias de estas técnicas:

1. **Técnica principal:** Usa una ventana de contexto muy grande (hasta 128K tokens en GPT-4o) y gestión de sesiones con base de datos, lo que le permite "recordar" conversaciones notablemente largas.

2. **Para conversaciones largas:** Cuando las conversaciones superan estos límites, emplea técnicas de ventana deslizante que priorizan:
   - El prompt de sistema e instrucciones
   - Los mensajes más recientes
   - Mensajes con alta densidad de información
   - Mensajes referenciados explícitamente por el usuario

3. **Compresión dinámica:** En algunas versiones, ChatGPT emplea algoritmos de compresión dinámica que resumen o eliminan selectivamente partes de la conversación menos relevantes para el intercambio actual.

Aunque estas soluciones ayudan, sigue existiendo un límite absoluto de lo que cualquier modelo puede "recordar" en una sola conversación. Por eso, incluso ChatGPT a veces "olvida" cosas mencionadas mucho antes en conversaciones muy largas.

## 9. Ventajas de los LLMs Locales

Usar LLMs locales con herramientas como Ollama tiene varias ventajas:

1. **Privacidad**: Tus datos nunca salen de tu computador
2. **Sin costos de API**: Ejecuta el modelo tanto como quieras sin pagar por consulta
3. **Uso sin conexión**: No necesitas internet una vez que el modelo está descargado
4. **Sin límites de uso**: Ejecuta tantas consultas como tu hardware pueda manejar

## 10. Recursos para Seguir Aprendiendo

- [Documentación de Ollama](https://github.com/ollama/ollama/blob/main/README.md)
- [Documentación de LangChain](https://python.langchain.com/docs/get_started/introduction)
- [Información sobre TinyLlama](https://github.com/jzhang38/TinyLlama)
"""

## Revisión crítica de las respuestas reales

Se probaron todas las llamadas con TinyLlama 1.1B local. Sin historial, el modelo inventó preferencias por azul, verde y amarillo: no es evidencia de memoria. Con historial mencionó azul, pero fue repetitivo y poco conciso.

El código de palíndromos generado **es incorrecto**: contiene un `if` con sintaxis inválida y mezcla la variable `I` con `i`. El ejemplo posterior también declara una función con un literal como parámetro. Por tanto, no puede ejecutarse como está. Una corrección para comparación literal sin distinguir mayúsculas es `s.lower() == s.lower()[::-1]`. No se ejecuta automáticamente código generado por el modelo. La primera respuesta enumera recursos de programación, repite Udacity y contiene descripciones que no se verificaron; se conserva como salida del modelo, no como recomendación validada.

Esto muestra por qué recibir una respuesta sin errores de API no basta: hay que revisar contenido y comprobar el código con casos que puedan refutarlo.